# 06 — Optimization Engine
**Levante Ferries · portfolio ficticio · 2026**

Ejecutar desde la raíz del proyecto o desde `notebooks`. Cada tabla se calcula sobre CSV regenerables.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import numpy as np
from model import assess,aggregate,calculate,optimize,FUELEU_LIMIT
DATA=ROOT/'data'; OUT=ROOT/'outputs'
voyages=pd.read_csv(DATA/'voyages_2026_synthetic.csv')
print('Raíz:',ROOT.name,'| viajes:',len(voyages),'| periodo:',voyages.date.min(),voyages.date.max())

Raíz: levante_ferries_sustainability | viajes: 4860 | periodo: 2026-01-01 2026-08-31


## Propósito
Problema binario por grupo ruta–buque: escoger exactamente una opción entre velocidades, mezcla bio y OPS. Se preserva el número de viajes, demanda y capacidades históricas, se respeta el tiempo máximo de mar, se impone un techo de CO₂e 12 % inferior a base y el objetivo de intensidad FuelEU proxy por buque. Minimizar coste fijo + energía + EUA, con ingreso fijo para comparar margen. No incluye CAPEX ni coste de sanción.

In [2]:
sel,summary=optimize(voyages)
print(summary)
print(sel[['route_id','vessel_id','voyages','speed_factor','bio_share','ops_share']].to_string(index=False))

{'solver_status': 0, 'solver_message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)', 'baseline_ttw_t': 40355.73092494059, 'optimized_ttw_t': np.float64(34953.92551855439), 'baseline_cost_eur': np.float64(41808105.562094115), 'optimized_cost_eur': np.float64(40576118.000078656), 'savings_eur': np.float64(1231987.562015459), 'cap_reduction': 0.12, 'groups': 10, 'candidates': 540}
route_id vessel_id  voyages  speed_factor  bio_share  ops_share
 ALG-TNG     LF-09      486          0.94       0.15        0.0
 BCN-IBZ     LF-07      486          0.94       0.00        0.7
 BCN-PMI     LF-08      486          0.94       0.00        0.0
 DNI-FOR     LF-03      486          0.94       0.15        0.0
 DNI-IBZ     LF-01      486          0.94       0.00        0.0
 DNI-IBZ     LF-02      486          0.94       0.00        0.7
 DNI-PMI     LF-04      486          0.94       0.00        0.0
 VLC-ORN     LF-10      486          0.94       0.15        0.0
 VLC-PMI     LF-05    

In [3]:
assert sel.voyages.sum()==len(voyages)
assert sel.ttw_kg.sum()<=summary['baseline_ttw_t']*1000*(1-summary['cap_reduction'])+1e-5
for _,g in sel.groupby('vessel_id'):
 assert g.fueleu_wtw_kg.sum()<=FUELEU_LIMIT*g.fueleu_energy_mj.sum()/1000+1e-5
print('Restricciones de frecuencia, emisiones e intensidad por buque verificadas')

Restricciones de frecuencia, emisiones e intensidad por buque verificadas


**Límite de solver:** solución MIP con tolerancia 0,5 % y límite 35 s; si reporta estado de tiempo, se utiliza la mejor solución factible y se explicita el estado.